# Caviar Strategy Comparison
Compares CSV result files produced by different proving strategies.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from pathlib import Path

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## Configuration
Edit this list to add/remove strategies. Each entry is `(filename, display_name)`.
Paths are relative to `../tmp/`.

In [ ]:
TMP = Path("../tmp")

CONFIGS: list[tuple[str, str]] = [
    ("detour_offset_3.csv", "Detour offset=3"),
    # ("detour_offset_5.csv",   "Detour offset=5"),
    # ("results_prove.csv",     "Simple"),
    # ("results_fast.csv",      "NPP"),
    # ("results_beh_0.5.csv",   "Pulse t=0.5"),
]

## Load data

In [ ]:
SCHEMA = {
    "index": pl.Int32,
    "start_expression": pl.String,
    "end_expression": pl.String,
    "result": pl.Boolean,
    "best_expr": pl.String,
    "class": pl.Int64,
    "iterations": pl.Int64,
    "egraph_size": pl.Int64,
    "rebuilds": pl.Int64,
    "total_time": pl.Float64,
    "stop_reason": pl.String,
    "condition": pl.String,
    "halide_result": pl.String,
    "halide_time": pl.Float64,
}

frames: dict[str, pl.DataFrame] = {}
for fname, label in CONFIGS:
    path = TMP / fname
    if not path.exists():
        print(f"WARNING: {path} not found, skipping")
        continue
    df = pl.read_csv(path, schema_overrides=SCHEMA, null_values=["", "null", "NULL"])
    frames[label] = df
    print(f"{label:30s}  {len(df):>6} rows  proved={df['result'].sum()}")

assert frames, "No data loaded — check CONFIGS paths"

## 1 · Prove rate

In [ ]:
labels = list(frames.keys())
rates = [float(frames[l]["result"].mean() or 0.0) * 100 for l in labels]
counts = [int(frames[l]["result"].sum()) for l in labels]
totals = [len(frames[l]) for l in labels]

fig, ax = plt.subplots(figsize=(max(5, len(labels) * 1.4), 4))
bars = ax.bar(labels, rates, color="steelblue", width=0.5)
for bar, c, t in zip(bars, counts, totals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{c}/{t}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("Prove rate (%)")
ax.set_ylim(0, 105)
ax.set_title("Prove rate by strategy")
ax.yaxis.set_major_formatter(ticker.PercentFormatter())
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 2 · Time to prove (successful only)

In [ ]:
fig, axes = plt.subplots(
    1, len(frames), figsize=(4 * len(frames), 4), sharey=False, squeeze=False
)
axes = axes[0]

for ax, (label, df) in zip(axes, frames.items()):
    proved = df.filter(pl.col("result"))["total_time"]
    if proved.is_empty():
        ax.text(
            0.5,
            0.5,
            "no proved\nexpressions",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    else:
        ax.hist(
            proved.to_numpy(),
            bins=30,
            color="seagreen",
            edgecolor="white",
            linewidth=0.4,
        )
        ax.axvline(
            proved.mean(),
            color="red",
            linestyle="--",
            linewidth=1,
            label=f"mean={proved.mean():.2f}s",
        )
        ax.legend(fontsize=8)
    ax.set_title(label)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Count")

fig.suptitle("Time to prove — successful expressions", fontsize=12)
plt.tight_layout()
plt.show()

## 3 · Total time distribution (all expressions — proved + timeout)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for label, df in frames.items():
    times = df["total_time"].to_numpy()
    ax.ecdf(times, label=label)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF of total_time (all expressions)")
ax.legend()
plt.tight_layout()
plt.show()

## 4 · Egraph size distribution

In [ ]:
fig, axes = plt.subplots(
    1, len(frames), figsize=(4 * len(frames), 4), sharey=False, squeeze=False
)
axes = axes[0]

for ax, (label, df) in zip(axes, frames.items()):
    sizes = df["egraph_size"].to_numpy()
    ax.hist(
        np.log10(sizes + 1),
        bins=30,
        color="mediumpurple",
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_title(label)
    ax.set_xlabel("log₁₀(egraph_size + 1)")
    ax.set_ylabel("Count")

fig.suptitle("Egraph size distribution", fontsize=12)
plt.tight_layout()
plt.show()

## 5 · Iterations distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for label, df in frames.items():
    iters = df["iterations"].to_numpy()
    ax.ecdf(iters, label=label)

ax.set_xlabel("Iterations")
ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF of iterations")
ax.legend()
plt.tight_layout()
plt.show()

## 6 · Stop reason breakdown

In [ ]:
# Collect all unique stop reasons across all files
all_reasons = sorted(
    set(
        r
        for df in frames.values()
        for r in df["stop_reason"].drop_nulls().unique().to_list()
    )
)

reason_data = {
    label: {r: int(df.filter(pl.col("stop_reason") == r).height) for r in all_reasons}
    for label, df in frames.items()
}

x = np.arange(len(labels))
width = 0.7 / max(len(all_reasons), 1)
colors = plt.colormaps["tab10"].colors  # type: ignore[attr-defined]

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.8), 4))
for i, reason in enumerate(all_reasons):
    counts = [reason_data[l].get(reason, 0) for l in labels]
    offset = (i - len(all_reasons) / 2 + 0.5) * width
    ax.bar(
        x + offset,
        counts,
        width=width * 0.9,
        label=reason,
        color=colors[i % len(colors)],
    )

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("Count")
ax.set_title("Stop reason breakdown")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 7 · Summary table

In [ ]:
rows = []
for label, df in frames.items():
    proved = df.filter(pl.col("result"))
    rows.append(
        {
            "strategy": label,
            "n": len(df),
            "proved": int(df["result"].sum()),
            "prove_rate_%": round(float(df["result"].mean() or 0.0) * 100, 1),
            "time_proved_mean": round(proved["total_time"].mean() or float("nan"), 3),
            "time_proved_med": round(proved["total_time"].median() or float("nan"), 3),
            "time_all_mean": round(df["total_time"].mean() or float("nan"), 3),
            "time_all_med": round(df["total_time"].median() or float("nan"), 3),
            "egraph_mean": round(
                df["egraph_size"].cast(pl.Float64).mean() or float("nan"), 1
            ),
            "iter_mean": round(
                df["iterations"].cast(pl.Float64).mean() or float("nan"), 1
            ),
        }
    )

summary = pl.DataFrame(rows)
print(summary)